In [1]:
import polars as pl
import numpy as np
import polars.selectors as cs

In [2]:
df = pl.read_parquet('../data/raw/tripdata.parquet')

Convert timestamps from String to Datetime

In [3]:
clean_df = df.with_columns(
    pl.col("started_at").str.to_datetime(),
    pl.col("ended_at").str.to_datetime(),
)

In [4]:
height_before = clean_df.height

clean_df = clean_df.unique()

print(f"{height_before - clean_df.height} duplicate records were excluded")

35 duplicate records were excluded


In [5]:
NULL_TOKENS = ["", "NA", "N/A", "null", "-"]

clean_df = clean_df.with_columns(
    pl.when(pl.col(pl.String).str.strip_chars().is_in(NULL_TOKENS))
      .then(None)
      .otherwise(pl.col(pl.String))
      .name.keep(),
    cs.float().fill_nan(None),
).with_columns(
    cs.string().str.strip_chars()
)

In [6]:
clean_df = clean_df.with_columns(
    pl.col("ended_at").sub(pl.col("started_at"))
    .dt.total_seconds()
    .alias("ride_length")
)

In [7]:
height_before = clean_df.height

clean_df = clean_df.filter(
    pl.col("ride_length") >= 60
)

print(f"{height_before - clean_df.height} rides less than 1 minute long were excluded")

159221 rides less than 1 minute long were excluded


`ride_length_hours_p0.9999` are basically the same for members and casual users, which reveals superficiality. It is safe to assume that we can exclude rides that last 24 hours or more.

In [8]:
(clean_df.with_columns(
    pl.col("ride_length").mul(1 / 3600).alias("ride_length_hours")
)
.group_by("member_casual")).agg(
    pl.col("ride_length_hours").quantile(q).name.suffix(f"_p{q}")
    for q in [.99, .999, .9999]
)

member_casual,ride_length_hours_p0.99,ride_length_hours_p0.999,ride_length_hours_p0.9999
str,f64,f64,f64
"""casual""",2.011944,24.997222,24.999167
"""member""",0.866111,4.011111,24.998056


In [9]:
height_before = clean_df.height

clean_df = clean_df.filter(
    pl.col("ride_length") < 60 * 60 * 24
)

print(f"{height_before - clean_df.height} rides higher than 24 hours long were excluded")

5317 rides higher than 24 hours long were excluded


In [10]:
height_before = clean_df.height

clean_df = clean_df.filter(
    pl.col("started_at") < pl.col("ended_at")
)

print(f"{height_before - clean_df.height} rides that start before they end were excluded")

0 rides that start before they end were excluded


Since records missing end_lng are also mission end station name and ID, then we cannot determine end station info. Therefore, we exclude those records.

In [11]:
height_before = clean_df.height

clean_df = clean_df.drop_nulls(subset=["end_lng"])

print(f"{height_before - clean_df.height} records without any end station info were excluded")

308 records without any end station info were excluded


In [12]:
def haversine(start_lat, start_long, end_lat, end_long):
     # Earth radius in kilometers
     R = 6371.0

     start_lat = pl.col(start_lat) * (np.pi / 180.0)
     start_long = pl.col(start_long) * (np.pi / 180.0)
     end_lat = pl.col(end_lat) * (np.pi / 180.0)
     end_long = pl.col(end_long) * (np.pi / 180.0)

     dlat = end_lat - start_lat
     dlong = end_long - start_long

     a = (
        (dlat / 2.0).sin() ** 2
        + start_lat.cos() * end_lat.cos() * (dlong / 2.0).sin() ** 2
     )
     c = 2 * np.arcsin(np.sqrt(a))

     return R * c

clean_df = clean_df.with_columns(
    pl.col("started_at").dt.weekday().alias("day_of_week"),
    pl.col("started_at").dt.hour().alias("start_hour"),
    pl.col("ended_at").dt.hour().alias("end_hour"),
    pl.col("started_at").dt.month().alias("month"),
    pl.col("started_at").dt.weekday().ge(6).alias("is_weekend"),
    pl.col("start_station_name").eq(pl.col("end_station_name")).alias("is_round_trip")
).with_columns(
    pl.when(pl.col("month").is_in([12, 1, 2])).then(pl.lit("Winter"))
    .when(pl.col("month").is_in([3, 4, 5])).then(pl.lit("Spring"))
    .when(pl.col("month").is_in([6, 7, 8])).then(pl.lit("Summer"))
    .otherwise(pl.lit("Autumn"))
    .alias("season")
).with_columns(
    distance_km=haversine("start_lat", "start_lng", "end_lat", "end_lng"),
)

In [13]:
# Considers a rectangular boundary around Chicago
LAT_MIN, LAT_MAX = 41.64, 42.03
LONG_MIN, LONG_MAX = -87.94, -87.52

clean_df = clean_df.with_columns(
    (pl.col("start_lat").is_between(LAT_MIN, LAT_MAX)
    & pl.col("start_lng").is_between(LONG_MIN, LONG_MAX))
    .alias("start_inside_chicago"),

    (pl.col("end_lat").is_between(LAT_MIN, LAT_MAX)
    & pl.col("end_lng").is_between(LONG_MIN, LONG_MAX))
    .alias("end_inside_chicago"),
)

In [14]:
clean_df = clean_df.with_columns(
    pl.col("start_station_name").is_not_null().alias("has_start_station"),
    pl.col("end_station_name").is_not_null().alias("has_end_station"),
)

In [15]:
print(f"{df.height - clean_df.height} records were excluded.\n"
      f"{clean_df.width - df.width} columns were added.")

164881 records were excluded.
13 columns were added.


In [16]:
def null_per_group(d: pl.DataFrame, group: str):
    return d.group_by(group).agg(
        pl.all().is_null().mean().mul(100).round(1).name.suffix("_pct_nulls"),
        pl.len().alias("n"),
    )

In [17]:
null_per_group(df, "member_casual")

member_casual,ride_id_pct_nulls,rideable_type_pct_nulls,started_at_pct_nulls,ended_at_pct_nulls,start_station_name_pct_nulls,start_station_id_pct_nulls,end_station_name_pct_nulls,end_station_id_pct_nulls,start_lat_pct_nulls,start_lng_pct_nulls,end_lat_pct_nulls,end_lng_pct_nulls,n
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""member""",0.0,0.0,0.0,0.0,20.4,20.4,20.3,20.3,0.0,0.0,0.0,0.0,3982761
"""casual""",0.0,0.0,0.0,0.0,21.8,21.8,25.0,25.0,0.0,0.0,0.2,0.2,2189024


In [18]:
null_per_group(clean_df, "member_casual")

member_casual,ride_id_pct_nulls,rideable_type_pct_nulls,started_at_pct_nulls,ended_at_pct_nulls,start_station_name_pct_nulls,start_station_id_pct_nulls,end_station_name_pct_nulls,end_station_id_pct_nulls,start_lat_pct_nulls,start_lng_pct_nulls,end_lat_pct_nulls,end_lng_pct_nulls,ride_length_pct_nulls,day_of_week_pct_nulls,start_hour_pct_nulls,end_hour_pct_nulls,month_pct_nulls,is_weekend_pct_nulls,is_round_trip_pct_nulls,season_pct_nulls,distance_km_pct_nulls,start_inside_chicago_pct_nulls,end_inside_chicago_pct_nulls,has_start_station_pct_nulls,has_end_station_pct_nulls,n
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""casual""",0.0,0.0,0.0,0.0,20.5,20.5,23.2,23.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.8,0.0,0.0,0.0,0.0,0.0,0.0,2110121
"""member""",0.0,0.0,0.0,0.0,19.8,19.8,19.4,19.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.9,0.0,0.0,0.0,0.0,0.0,0.0,3896783


In [19]:
clean_df["member_casual"].value_counts(normalize=True)

member_casual,proportion
str,f64
"""casual""",0.351283
"""member""",0.648717


In [20]:
clean_df[["ride_id", "started_at", "ended_at", "rideable_type", "member_casual"]].null_count()

ride_id,started_at,ended_at,rideable_type,member_casual
u32,u32,u32,u32,u32
0,0,0,0,0


In [21]:
clean_df["ride_length"].describe()

statistic,value
str,f64
"""count""",6.006904e6
"""null_count""",0.0
"""mean""",855.390892
"""std""",1612.84099
"""min""",60.0
"""25%""",334.0
"""50%""",566.0
"""75%""",977.0
"""max""",86398.0


In [23]:
clean_df.write_parquet("../data/processed/clean_tripdata.parquet", mkdir=True)